# Lab 2 (Iris Version) - Implementing k-Nearest Neighbours for Classification

This notebook mirrors the structure of the original k-NN lab, but uses the **Iris dataset**
instead of the Breast Cancer dataset for the real-world classification example.

We will:
1. Load and explore the Iris dataset
2. Split it into training and test sets
3. Train a `KNeighborsClassifier`
4. Evaluate accuracy on the test set
5. Visualize decision boundaries (using two features)
6. Study model complexity (train vs test accuracy across different k values)

## The Iris Dataset
The Iris dataset is a classic real-world dataset for classification.
It contains 150 samples of iris flowers from **3 species** (setosa, versicolor, virginica),
described by 4 features: sepal length, sepal width, petal length, and petal width.

The data can be loaded using the `load_iris` function from scikit-learn (same way `load_breast_cancer` was used).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_iris

iris = load_iris()
print(f"iris.keys():\n{iris.keys()}")

In [ ]:
print(iris["DESCR"][:1000])  # first part of the description

The dataset consists of 150 data points, with 4 features each:

In [ ]:
print(f"Shape of iris data: {iris.data.shape}")

There are 50 samples for each of the 3 classes:

In [ ]:
sample_counts = {n: v for n, v in zip(iris.target_names, np.bincount(iris.target))}
print(f"Sample counts per class:\n{sample_counts}")

Feature names:

In [ ]:
print(f"Feature names:\n{iris.feature_names}")

## k-Nearest Neighbours using Scikit-Learn

First, we split the data into a training set and a test set so we can evaluate
generalization performance.

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    iris.data, iris.target, stratify=iris.target, random_state=0)

Next, we import and instantiate the `KNeighborsClassifier`.

This is when we can set parameters, like the number of neighbors to use.

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

clf = KNeighborsClassifier(n_neighbors=3)

Now we fit the classifier using the training set (this stores the training data for later use during prediction):

In [ ]:
clf.fit(X_train, y_train)

To make predictions on the test data, we call the `predict` method.

For each data point in the test set, this computes its nearest neighbors in the training set and finds the most common class among these:

In [ ]:
print(f"Test set predictions: {clf.predict(X_test)}")

To evaluate how well our model generalizes, we call the `score` method with the test data together with the test labels:

In [ ]:
print(f"Test set accuracy: {clf.score(X_test, y_test):.2f}")

## Analysing the k-NN Classifier's Decision Boundary

Since Iris has 4 features, we can't easily plot a decision boundary in 4D.
To visualize the decision boundary like we did with the 2D forge dataset, we will
use just the first two features (sepal length and sepal width).

In [ ]:
import mglearn

# use only the first two features so we can visualize in 2D
X2 = iris.data[:, :2]
y2 = iris.target

X2_train, X2_test, y2_train, y2_test = train_test_split(
    X2, y2, stratify=y2, random_state=0)

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for n_neighbors, ax in zip([1, 3, 9], axes):
    clf2 = KNeighborsClassifier(n_neighbors=n_neighbors).fit(X2_train, y2_train)
    mglearn.plots.plot_2d_separator(clf2, X2, fill=True, eps=0.5, ax=ax, alpha=.4)
    mglearn.discrete_scatter(X2[:, 0], X2[:, 1], y2, ax=ax)
    ax.set_title(f"{n_neighbors} neighbor(s)")
    ax.set_xlabel(iris.feature_names[0])
    ax.set_ylabel(iris.feature_names[1])

axes[0].legend(loc=3)

Just like with the forge dataset, using a single neighbor gives a decision boundary
that closely follows the training points (high complexity), while using more neighbors
gives a smoother boundary (lower complexity).

## Investigating the Connection Between Model Complexity and Generalization

We repeat the same experiment done on the Breast Cancer dataset, but now using the
full 4-feature Iris dataset. We split into train/test sets and evaluate training and
test accuracy across a range of `n_neighbors` values.

In [ ]:
training_accuracy = []
test_accuracy = []
# try n_neighbors from 1 to 10
neighbors_settings = range(1, 11)

for n_neighbors in neighbors_settings:
    clf = KNeighborsClassifier(n_neighbors=n_neighbors)
    clf.fit(X_train, y_train)
    # record training set accuracy
    training_accuracy.append(clf.score(X_train, y_train))
    # record generalization accuracy
    test_accuracy.append(clf.score(X_test, y_test))

plt.plot(neighbors_settings, training_accuracy, label="training accuracy")
plt.plot(neighbors_settings, test_accuracy, label="test accuracy")
plt.ylabel("Accuracy")
plt.xlabel("n_neighbors")
plt.legend()

As with the Breast Cancer example:
* A single neighbor (k=1) gives perfect (or near-perfect) training accuracy, since each
  point's nearest neighbor is itself.
* As k increases, the model becomes simpler (smoother decision boundary), and training
  accuracy decreases.
* Test accuracy typically peaks somewhere in the middle range of k, and can drop again
  for very large k, where the model becomes too simple (underfitting).

Because Iris is a relatively small and easy (linearly well-separated) dataset, you will
likely see very high test accuracy (often 90%+) across a wide range of k values.

## Strengths, Weaknesses, and Parameters (Recap)
* The two important parameters of k-NN are the **number of neighbors (k)** and the
  **distance metric** used (Euclidean by default).
* A small k (e.g. 1) leads to a high-complexity model that can overfit.
* A large k leads to a low-complexity model that can underfit.
* k-NN is simple to understand and often gives a strong baseline, but can be slow on
  large datasets and does not perform well with many features or sparse data.
* As always, it's good practice to preprocess/scale your features before using k-NN,
  since it relies on distance calculations.